# C4 Model Creation
This notebook generates a C4 metabolic model starting from a mass and proton-balanced model of plant primary metabolism. It involves curating the base model, duplicating it to represent Mesophyll and Bundle Sheath cells, and applying specific physiological constraints.

## 1. Setup and Initialization
Import necessary libraries and load the generic core plant metabolism model.

In [1]:
import pandas as pd
import sys
from cobra.io import read_sbml_model, write_sbml_model
from cobra import Reaction
sys.path.append("../Code")
from model_functions import *

# Load the General Core Model (PlantCoreMetabolism)
general_model = read_sbml_model("../Models/PlantCoreMetabolism_v2_0_0.xml")

# Keep a reference to the previous model version if needed (optional)
previous_model = read_sbml_model("../Models/PlantCoreMetabolism_v2_0_0.xml")

Set parameter Username
Academic license - for non-commercial use only - expires 2027-01-20


## 2. Model Curation: Adding Reactions
In this section, we add specific biomass reactions and transporters required for the analysis that were not present in the base model.

### 2.1 Biomass Reactions
Define biomass compositions based on simplified datasets.

In [2]:
# Create the biomass reaction object
reaction = Reaction('Arabidopsis_biomass_tx')
reaction.name = 'Arabidopsis biomass'
reaction.lower_bound = 0.
reaction.upper_bound = 1000.

# Import biomass composition data
df = pd.read_csv("../Datasets/Biomass_Simplified.csv")

# Retrieve metabolite objects
met_list = []
for id in df.loc[:,"Ids"]:
    met = general_model.metabolites.get_by_id(id)
    met_list.append(met)

# Get coefficients for Arabidopsis
coef = list(df.loc[:,"Arabidopsis"])

# Add metabolites to the reaction
for i in range(len(coef)):
    reaction.add_metabolites({met_list[i]:coef[i]})

# Add the reaction to the model
general_model.add_reactions([reaction])

### 2.2 Transporters
Add specific intracellular transporters for Malate, Pyruvate, and PEP.

In [3]:
# --- Malate/Pyruvate Transporter ---
# Define metabolites involved
MAL_c = general_model.metabolites.MAL_c
MAL_p = general_model.metabolites.MAL_p
PYRUVATE_c = general_model.metabolites.PYRUVATE_c
PYRUVATE_p = general_model.metabolites.PYRUVATE_p

# Create the transporter reaction
transporter = Reaction('PYR_MAL_pc')
transporter.name = 'Malate/Pyruvate Transport'
transporter.lower_bound = -1000.
transporter.upper_bound = 1000.

# Add stoichiometry (Antiport)
transporter.add_metabolites({
    PYRUVATE_p: -1.0,
    MAL_c: -1.0,
    PYRUVATE_c: 1.0,
    MAL_p: 1.0
})

general_model.add_reactions([transporter])

In [4]:
# --- Proton Mediated Pyruvate Transporter ---
Pyr_c = general_model.metabolites.PYRUVATE_c
H_c = general_model.metabolites.PROTON_c
Pyr_p = general_model.metabolites.PYRUVATE_p
H_p = general_model.metabolites.PROTON_p

transporter = Reaction('Pyr_H_pc')
transporter.name = 'Proton Mediated Pyruvate Transport'
transporter.lower_bound = -1000.
transporter.upper_bound = 1000.

# Symport with proton
transporter.add_metabolites({
    Pyr_c: -1.0,
    H_c: -1.0,
    Pyr_p: 1.0,
    H_p: 1.0
})

general_model.add_reactions([transporter])

In [5]:
# --- PEP Transporter (Corrected Proton Stoichiometry) ---
H_c = general_model.metabolites.PROTON_c
H_p = general_model.metabolites.PROTON_p
PI_c = general_model.metabolites.Pi_c
PI_p = general_model.metabolites.Pi_p
PEP_c = general_model.metabolites.PHOSPHO_ENOL_PYRUVATE_c
PEP_p = general_model.metabolites.PHOSPHO_ENOL_PYRUVATE_p

transporter = Reaction('PPT_pc')
transporter.name = 'Phosphoenolpyruvate Transporter'
transporter.lower_bound = -1000.
transporter.upper_bound = 1000.

transporter.add_metabolites({
    PEP_p: -1.0,
    H_p: -1.0,
    PI_c: -1,
    PEP_c: 1.0,
    H_c: 1.0,
    PI_p: 1.0
})

general_model.add_reactions([transporter])

## 3. Initial Constraints for Base Model
Apply generic constraints to the base model, such as defining import/export bounds and blocking specific reactions.

In [6]:
# --- Generic Constraints ---

# Initial import/export constraints
set_bounds('CO2_tx', (-1000, 1000), general_model)
set_bounds('H2O_tx', (-1000, 1000), general_model)
set_bounds('NH4_tx', (0., 0.), general_model)
set_bounds('Pi_tx', (0, 1000), general_model)
set_bounds('SO4_tx', (0, 1000), general_model)
set_bounds('O2_tx', (-1000, 1000), general_model)
set_bounds('Sucrose_tx', (-1000, 0), general_model) # Exported but not imported
set_bounds('GLC_tx', (-1000, 0), general_model)
set_bounds('unlProtHYPO_c', (0, 0), general_model)
set_bounds('H_tx', (0, 0), general_model)

# Bounds for ATP maintenance
set_bounds('ATPase_tx', (0, 1000), general_model)

# Bounds for chloroplastic NADPH dehydrogenase and plastoquinol oxidase
set_bounds('Plastoquinol_Oxidase_p', (0, 0), general_model)

# NTT is only active at night
set_fixed_flux("ATP_ADP_Pi_pc", 0, general_model)

# No uncoupled pyruvate transport
set_bounds('PYRUVATE_pc', (0, 0), general_model)

# Blocking Extra Biomass reaction
set_bounds("AraCore_Biomass_tx", (0,1000), general_model)
set_bounds("Phloem_output_tx", (0,0), general_model)

# Block Former PEP transport reaction
set_bounds("PEP_Pi_pc", (0,0), general_model)

## 4. Generation of the C4 Model
This section creates the two-cell C4 model by duplicating the base model for Mesophyll (M) and Bundle Sheath (B) cells and establishing connections between them.

### 4.1 Model Duplication
Create a new model `c4_model` and populate it with metabolites and reactions prefixed with `[M]_` and `[B]_`.

In [7]:
from cobra import Model, Reaction, Metabolite

c4_model = Model('c4_model')
cell_types = ['M', 'B']

# --- Duplicate Metabolites ---
for m in general_model.metabolites:
    for cell in cell_types:
        m_dt = Metabolite('['+cell+']_'+m.id, name= m.name,
            formula     = m.formula,
            compartment = m.compartment
        )
        c4_model.add_metabolites([m_dt])

# --- Duplicate Reactions ---
for r_c3_obj in general_model.reactions:
    for cell in cell_types:
        r_c4_obj = Reaction('['+cell+']_'+r_c3_obj.id)
        r_c4_obj.name = r_c3_obj.name
        r_c4_obj.subsystem = r_c3_obj.subsystem
        r_c4_obj.bounds = r_c3_obj.bounds
        c4_model.add_reactions([r_c4_obj])
        # Add metabolites with correct coefficients to the new reaction
        r_c4_obj.add_metabolites({'['+cell+']_'+m_c3_obj.id: r_c3_obj.get_coefficient(m_c3_obj) for m_c3_obj in r_c3_obj.metabolites})

# Set objective function
c4_model.objective = '[M]_Arabidopsis_biomass_tx'

### 4.2 Intercellular Transport (Mesophyll <-> Bundle Sheath)
Establish exchange reactions for cytosolic metabolites between the two cell types, excluding specific metabolites that do not transport.

In [8]:
# Define metabolites that should NOT be transported between cells
Na,F26BP, ORO, DHO, GABA, PRPP, AD, DHF, starch, TRXox, TRXrd, T6P, OMP, UMP, CTP, dADP, dCDP, dGDP, dUDP, dATP, dCTP, dGTP = 0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0

no_transport_general = [ Na, "HS",
                        "FRUCTOSE_16_DIPHOSPHATE",  F26BP, "DPG", "PROTON", "ACETALD", "ACET", "5_10_METHENYL_THF",
                        "5_METHYL_THF", "HOMO_CYS", "ADENOSYL_HOMO_CYS", ORO, DHO,
                        GABA, "ACETYLSERINE", PRPP, AD, "THF", DHF , "ADENOSINE", "MALTOSE", "CO_A", "L_GLUTAMATE_5_P", "aL_GLUTAMATE_5_P",
                        "ACETYL_COA", "CELLULOSE", starch, TRXox, TRXrd, "L_GLUTAMATE_GAMMA_SEMIALDEHYDE", T6P, "S_ADENOSYLMETHIONINE",
                        "PPI", "L_DELTA1_PYRROLINE_5_CARBOXYLATE", "CARBON_DIOXIDE", "OXALACETIC_ACID", "HCO3",
                        "UTP","aUTP", "UDP","aUDP", "UDP_GLUCOSE", "ATP", "aATP","ADP", "aADP","AMP", "IMP", "aIMP","XANTHOSINE_5_PHOSPHATE", "aXANTHOSINE_5_PHOSPHATE",
                        "GTP", "aGTP","GDP","aGDP", "GMP","aGMP", "bGMP",OMP, UMP, CTP, "GDP", "aGDP","CDP", dADP,
                        dCDP, dGDP, dUDP, "DUMP", "DTMP",  "aDTDP", "GTP",
                        dATP, dCTP, dGTP, "DTTP" ,"aDTTP", "NAD", "NADH", "NADP", "NADPH"]

# --- Add M/BS Exchange Reactions ---
L_r_transport = []
for m_c3_obj in general_model.metabolites:
    # Check if metabolite is cytosolic ('c') and not in the exclusion list
    if m_c3_obj.id[-1:] == 'c' and m_c3_obj.id[:-2] not in no_transport_general and  m_c3_obj.id[-2:] != 'mc':
        r_c4_obj = Reaction('[MB]_'+m_c3_obj.id)
        r_c4_obj.name = '[MB]_'+m_c3_obj.id
        r_c4_obj.subsystem = 'Exchange'
        r_c4_obj.bounds = (-1000, 1000)
        c4_model.add_reactions([r_c4_obj])
        # Reaction: [M]_Met <-> [B]_Met
        r_c4_obj.add_metabolites({'[M]_'+m_c3_obj.id: -1,'[B]_'+m_c3_obj.id: 1 })
        L_r_transport.append('[MB]_'+m_c3_obj.id)

In [9]:
# M–BS exchange/transport metabolites
n_exchange_mets = len(L_r_transport)

print(f"Number of exchange metabolites: {n_exchange_mets}")

Number of exchange metabolites: 194


### 4.3 Rubisco and CO2 Handling
Configure Rubisco in the Bundle Sheath to use a distinct "external" CO2 species to model the concentration mechanism.

In [10]:
# --- Add external CO2 species to Bundle Sheath ---
# (The original CO2 species is treated as internal CO2)
m_list_CO_Ex= ['[B]_CARBON_DIOXIDE_ex_c','[B]_CARBON_DIOXIDE_ex_p']

for m_id in m_list_CO_Ex:
    m_obj = Metabolite(m_id)
    m_obj.compartment = m_id[-1]
    c4_model.add_metabolites(m_obj)

# --- Duplicate reactions for external CO2 ---
# Copy reactions 'Tr_CO2h', 'RBC_h' and replace internal CO2 with external CO2
r_list_CO_Ex = ['CO2_pc', 'RIBULOSE_BISPHOSPHATE_CARBOXYLASE_RXN_p']

for r_id in r_list_CO_Ex:
    r_obj = c4_model.reactions.get_by_id('[B]_'+r_id)
    r_obj_Ex = Reaction(r_obj.id+'_Ex')
    r_obj_Ex.name = r_obj.id+'_Ex'
    r_obj_Ex.subsystem = r_obj.subsystem
    r_obj_Ex.bounds = r_obj.bounds
    c4_model.add_reactions([r_obj_Ex])
    # Replace metabolite if it is CO2
    r_obj_Ex.add_metabolites({m_obj.id if not m_obj.id[:-2] == '[B]_CARBON_DIOXIDE' else '[B]_CARBON_DIOXIDE_ex'+m_obj.id[-2:]: r_obj.get_coefficient(m_obj)
                                  for m_obj in r_obj.metabolites})

# --- CO2 Exchange between Mesophyll and Bundle Sheath ---
r_c4_obj = Reaction('[MB]_CO2_c')
r_c4_obj.name = '[MB]_CO2_c'
r_c4_obj.subsystem = 'Exchange'
r_c4_obj.bounds = (-1000, 1000)
c4_model.add_reactions([r_c4_obj])
r_c4_obj.add_metabolites({'[M]_CARBON_DIOXIDE_c': -1,'[B]_CARBON_DIOXIDE_ex_c': 1 })
L_r_transport.append('[MB]_CARBON_DIOXIDE_c')

### 4.4 Leakage Reactions
Implement leakage of CO2 and O2 from the Bundle Sheath back to the Mesophyll.

In [11]:
# --- CO2 Leakage Reaction ---
M_CO2 = c4_model.metabolites.get_by_id('[M]_CARBON_DIOXIDE_c')
B_CO2 = c4_model.metabolites.get_by_id('[B]_CARBON_DIOXIDE_c')

leakage_CO2 = Reaction('[MB]_CO2_leakage_c')
leakage_CO2.name = 'CO2 Leakage from Bundle Sheath to Mesophyll'
leakage_CO2.lower_bound = 0
leakage_CO2.upper_bound = 0 # Initially closed

leakage_CO2.add_metabolites({
    M_CO2: -1.0,
    B_CO2: 1.0
})

c4_model.add_reactions([leakage_CO2])

In [12]:
# --- O2 Leakage Reaction ---
M_O2 = c4_model.metabolites.get_by_id('[M]_OXYGEN_MOLECULE_c')
B_O2 = c4_model.metabolites.get_by_id('[B]_OXYGEN_MOLECULE_c')

leakage_O2 = Reaction('[MB]_O2_leakage_c')
leakage_O2.name = 'O2 Leakage from Bundle Sheath to Mesophyll'
leakage_O2.lower_bound = -1000.
leakage_O2.upper_bound = 0

leakage_O2.add_metabolites({
    M_O2: -1.0,
    B_O2: 1.0
})

c4_model.add_reactions([leakage_O2])

## 5. C4 Specific Constraints
Apply physiological constraints specific to C4 photosynthesis, such as restricting gas exchange in the Bundle Sheath (due to the suberin layer) and forcing the NADP-ME decarboxylation pathway.

In [13]:
# --- C4 Specific Constraints ---

# No CO2 uptake in Bundle Sheath cells due to suberin layer in cell membranes
set_bounds("[B]_CO2_tx", (0.,0.), c4_model)

# Gaseous exchanges only through the M cell
set_bounds("[B]_O2_tx", (0.,0.,), c4_model)

# Only allow uptakes through the BS of inorganic metabolites (if applicable, here restricted in M)
# Note: The comments say "Only allow uptakes through the BS" but the code restricts [M] uptakes.
# This implies these specific nutrients enter via BS in this model configuration.
set_bounds("[M]_Nitrate_tx", (0.,0.), c4_model)
set_bounds("[M]_SO4_tx", (0.,0.), c4_model)
set_bounds("[M]_H2O_tx", (0.,0.), c4_model)
set_bounds("[M]_Ca_tx", (0.,0.), c4_model)
set_bounds("[M]_Mg_tx", (0.,0.), c4_model)
set_bounds("[M]_H2O_tx", (0.,0.), c4_model)
set_bounds("[M]_Pi_tx", (0.,0.), c4_model)
set_bounds("[M]_Ca_tx", (0.,0.), c4_model)
set_bounds("[M]_K_tx", (0.,0.), c4_model)

# Force NADP-ME decarboxylation pathway in the plastid
set_fixed_flux("[B]_MALIC_NADP_RXN_c", 0, c4_model)

# Force NADP-ME decarboxylation pathways: make alternative decarboxylation routes irreversible or blocked
set_bounds('[B]_CARBAMATE_KINASE_RXN_p', (0, 1000), c4_model)
set_bounds('[M]_CARBAMATE_KINASE_RXN_p', (0, 1000), c4_model)

set_bounds('[B]_UREASE_RXN_c', (0, 0), c4_model)
set_bounds('[M]_UREASE_RXN_c', (0, 99999999), c4_model)

set_bounds('[B]_ISOCITDEH_RXN_m', (0, 1000), c4_model)
set_bounds('[M]_ISOCITDEH_RXN_m', (0, 1000), c4_model)

set_bounds('[B]_ISOCITDEH_RXN_c', (0, 1000), c4_model)
set_bounds('[M]_ISOCITDEH_RXN_c', (0, 1000), c4_model)

set_bounds('[B]_ISOCITRATE_DEHYDROGENASE_NAD_RXN_m', (0, 1000), c4_model)
set_bounds('[M]_ISOCITRATE_DEHYDROGENASE_NAD_RXN_m', (0, 1000), c4_model)

set_bounds('[M]_PYRUVATEORTHOPHOSPHATE_DIKINASE_RXN_c', (0, 0), c4_model)

# Block PSII in the BS cell (characteristic of NADP-ME subtype)
set_bounds('[B]_PSII_RXN_p', (0, 0), c4_model)

# Make Ferrodoxin:NADP(H) oxidoreductase (FNR) irreversible
set_bounds("[B]_1_PERIOD_18_PERIOD_1_PERIOD_2_RXN_p", (0, 1000), c4_model)
set_bounds("[M]_1_PERIOD_18_PERIOD_1_PERIOD_2_RXN_p", (0, 1000), c4_model)

In [14]:
# Increase bounds for reactions with default limits to avoid artificial constraints
for reaction in c4_model.reactions:
    if reaction.lower_bound == -1000:
        reaction.lower_bound = -9999
    if reaction.upper_bound == 1000:
        reaction.upper_bound = 9999

## 6. Model Export
Save the curated C3 (general) and C4 models to SBML files.

In [15]:
# Write the models to XML
write_sbml_model(general_model, "../Models/c3_model.xml")
write_sbml_model(c4_model, "../Models/c4_model.xml")